In [1]:
import os
import torch
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
from torchvision import transforms
from torchvision.transforms import v2
from torchvision import tv_tensors
from PIL import Image
import torchvision
from torch.optim import AdamW
from torch.optim import lr_scheduler
from torch.amp import autocast
from torch.nn.utils.rnn import pad_sequence
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

from Swin_classif import SwinTransformerClassification
from Swin_OD import SwinTransformerDetection


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# Dataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Download images and annotations for PASCAL VOC 2007 and 2012
voc_2007_train = torchvision.datasets.VOCDetection(
    root='/content/voc_data',
    year='2007',
    image_set='trainval',
    download=True
)

voc_2012_train = torchvision.datasets.VOCDetection(
    root='/content/voc_data',
    year='2012',
    image_set='trainval',
    download=True
)

voc_2007_test = torchvision.datasets.VOCDetection(
    root='/content/voc_data',
    year='2007',
    image_set='test',
    download=True
)

100%|██████████| 460M/460M [00:22<00:00, 20.5MB/s]
100%|██████████| 2.00G/2.00G [07:31<00:00, 4.43MB/s]
100%|██████████| 451M/451M [00:24<00:00, 18.2MB/s]


In [4]:
from torchvision.transforms import v2
from torchvision import tv_tensors
from PIL import Image

class PascalVocDataset(Dataset):
  def __init__(self, pascal_voc_dataset: torchvision.datasets.voc.VOCDetection, mapping_class_id, split="train", image_size=(224, 224)):
    super(PascalVocDataset, self).__init__()
    self.pascal_voc_dataset = pascal_voc_dataset
    self.image_size = image_size
    self.mapping_class_id = mapping_class_id
    self.split = split

    # Prétraitement des annotations pour éviter de toucher au disque/XML après
    self.annotations = []
    print("Pré-chargement des annotations en mémoire...")
    for i in range(len(pascal_voc_dataset)):
        # On ne récupère que l'annotation, pas l'image ici
        # torchvision lit le XML ici
        _, meta = pascal_voc_dataset[i]
        self.annotations.append(meta["annotation"])

    # Object Detection V2 transforms pipeline
    if self.split == "train":
        self.transform = v2.Compose([
            v2.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
            v2.RandomHorizontalFlip(p=0.5), # Automatically flips bounding boxes too
            v2.Resize(self.image_size),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        self.transform = v2.Compose([
            v2.Resize(self.image_size),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

  def __len__(self):
    return self.pascal_voc_dataset.__len__()

  def __getitem__(self, idx):
    img_path = self.pascal_voc_dataset.images[idx]
    image = Image.open(img_path).convert("RGB")

    annotation = self.annotations[idx]
    filename = annotation["filename"]

    # Original un-resized sizes
    size = annotation["size"]
    width, height = int(size["width"]), int(size["height"])

    objects = annotation.get("object", [])

    bounding_boxes = []
    labels = []

    if objects:
        for obj in objects:
            class_name = obj["name"]
            bnbox = obj["bndbox"]

            # Extract basic coordinates as xmin, ymin, xmax, ymax (Standard Format for tv_tensors)
            xmin, ymin = float(bnbox["xmin"]), float(bnbox["ymin"])
            xmax, ymax = float(bnbox["xmax"]), float(bnbox["ymax"])

            # Sanity check: Ensure positive width and height to avoid NaN later
            if xmax <= xmin or ymax <= ymin:
                continue

            label = self.mapping_class_id[class_name]
            bounding_boxes.append([xmin, ymin, xmax, ymax])
            labels.append(label)

        if len(bounding_boxes) == 0: # If all boxes were invalid
             return self.get_empty_item(filename, image)

        # Wrap bounding boxes in a tv_tensor so transform updates them exactly like the image
        tv_boxes = tv_tensors.BoundingBoxes(
            bounding_boxes,
            format=tv_tensors.BoundingBoxFormat.XYXY,
            canvas_size=(height, width),
            dtype=torch.float32
        )

        # Apply all transforms (Image and Boxes simultaneously)
        image, tv_boxes = self.transform(image, tv_boxes)

        # Convert transformed bounding boxes to expected model format [ymin, xmin, height, width]
        final_bounding_boxes = []
        for box in tv_boxes:
            xmin, ymin, xmax, ymax = box.tolist()

            # Robustness: constrain coordinates to the absolute image bounds and ensure width/height > 0
            xmin = max(0, min(xmin, self.image_size[1] - 1))
            ymin = max(0, min(ymin, self.image_size[0] - 1))
            xmax = max(xmin + 1, min(xmax, self.image_size[1]))
            ymax = max(ymin + 1, min(ymax, self.image_size[0]))

            final_bounding_boxes.append(torch.tensor([int(ymin), int(xmin), int(ymax-ymin), int(xmax-xmin)], dtype=torch.long))

        labels = [torch.tensor(l, dtype=torch.long) for l in labels]

    else:
        return self.get_empty_item(filename, image)

    return filename, image, torch.stack(final_bounding_boxes), torch.stack(labels)

  def get_empty_item(self, filename, image):
      # Handle images with NO items
      image = self.transform(image)
      final_bounding_boxes = [torch.tensor([-1, -1, -1, -1], dtype=torch.long)]
      labels = [torch.tensor(-1, dtype=torch.long)]
      return filename, image, torch.stack(final_bounding_boxes), torch.stack(labels)

In [5]:
def custom_collate_fn(batch):
    batch = [item for item in batch if item[0] is not None]
    if not batch:
        return None, None

    filenames, images, bounding_boxes, labels = zip(*batch)

    # need to pad to the image with the maximum bounding boxes
    bounding_boxes = pad_sequence(bounding_boxes, padding_value=-1, batch_first=True)
    labels = pad_sequence(labels, padding_value=-1, batch_first=True)

    return filenames, torch.stack(images), bounding_boxes, labels

In [6]:
# training data
torch.manual_seed(42)
batch_size = 64
mapping_class_id = {"background":0, "person":1, "bird":2, "cat":3, "cow":4, "dog":5, "horse":6, "sheep":7, "aeroplane":8, "bicycle":9, "boat":10, "bus":11, "car":12, "motorbike":13, "train":14, "bottle":15, "chair":16, "diningtable":17, "pottedplant":18, "sofa":19, "tvmonitor":20}
mapping_id_class = {}
for key, value in mapping_class_id.items():
  mapping_id_class[value] = key

training_dataset_2007 = PascalVocDataset(pascal_voc_dataset=voc_2007_train, mapping_class_id=mapping_class_id, image_size=(512,512))
training_dataset_2012 = PascalVocDataset(pascal_voc_dataset=voc_2012_train, mapping_class_id=mapping_class_id, image_size=(512,512))
dataset = ConcatDataset([training_dataset_2007, training_dataset_2012])

dataset_size = len(dataset)
train_size = int(0.8 * dataset_size)
val_size = dataset_size - train_size

training_dataset, validating_dataset = random_split(dataset, [train_size, val_size])

test_dataset = PascalVocDataset(pascal_voc_dataset=voc_2007_test, mapping_class_id=mapping_class_id, image_size=(512,512))

training_dataloader = DataLoader(training_dataset, batch_size = batch_size, shuffle=True, collate_fn = custom_collate_fn, drop_last=True, num_workers=2)
validating_dataloader = DataLoader(validating_dataset, batch_size = batch_size, shuffle=False,  collate_fn = custom_collate_fn, drop_last=False, num_workers=2)
testing_dataloader = DataLoader(test_dataset, batch_size = batch_size, shuffle=False, collate_fn = custom_collate_fn, drop_last=False, num_workers=2)

Pré-chargement des annotations en mémoire...
Pré-chargement des annotations en mémoire...
Pré-chargement des annotations en mémoire...


# Model

### Retrieve the classification pretrained weigth on ImageNet100

In [7]:
# model_classif = SwinTransformerClassification(
#     backbone_weights_path = None,
#     backbone_input_size = (224, 224),
#     backbone_patch_size = 4,
#     backbone_patch_merging_ratio = 2,
#     backbone_in_channels = 3,
#     backbone_layers = [2, 2, 2, 2],
#     backbone_query_size = 32,
#     backbone_n_heads = [2, 4, 8, 16],
#     backbone_mlp_factor=4,
#     backbone_window_size = 7,
#     n_classes = 100,
# )
# model_classif.load_state_dict(torch.load("/content/drive/MyDrive/output_model/Swin_classif_epoch_36.pt"))
# backbone_weights_path = "/content/drive/MyDrive/output_model/Swin_pretrained_backbone_weight.pt"
# torch.save(model_classif.backbone.state_dict(), backbone_weights_path)

In [7]:
backbone_weights_path = "/content/drive/MyDrive/output_model/Swin_pretrained_backbone_weight.pt"

model = SwinTransformerDetection(
    backbone_weights_path = backbone_weights_path,
    backbone_patch_size = 4,
    backbone_patch_merging_ratio = 2,
    backbone_in_channels = 3,
    backbone_layers = [2, 2, 2, 2],
    backbone_query_size = 32,
    backbone_n_heads = [2, 4, 8, 16],
    backbone_mlp_factor=4,
    backbone_window_size = 7,
    head_weights_path = None,
    head_scales = [64, 128, 256],
    head_shapes = [[1,1],[1,2],[2,1]],
    head_fixed_shape = [16, 16, [32, 32]],
    head_objectness_iou_threshold_positive_boxes = 0.7,
    head_objectness_iou_threshold_negative_boxes = 0.3,
    head_xywh_format = False,
    head_hidden_dim = 1024,
    n_classes = 20,
    background_label_id = 0
).to(device)

/content/Faster_RCNN.py:58: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  length = torch.tensor(scale * shape_tensor)


Backbone weights loaded from /content/drive/MyDrive/output_model/Swin_pretrained_backbone_weight.pt


# TRAINING

In [9]:
def train_one_epoch(i, model, device, train_loader, optimizer, scheduler, output_dir, accumulation_step=8):
    model.train()
    losses_displayed = []
    total_losses = []
    step = 0
    optimizer.zero_grad()

    for _, image_src, bounding_boxes, labels in train_loader:
        step += 1

        image_src = image_src.to(device)
        bounding_boxes  = bounding_boxes.to(device)
        labels = labels.to(device)

        with autocast(device_type=device, dtype=torch.bfloat16):
            loss, _, _, _, _ = model.forward_train(image_src, bounding_boxes, labels)
        loss = loss / accumulation_step
        loss.backward()
        if step%accumulation_step==0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        losses_displayed.append(loss.item())
        total_losses.append(loss.item())

        if step%20 == 0:
            print(f"training loss mean on last 20 steps : {str(sum(losses_displayed)/len(losses_displayed))}")
            losses_displayed = []

    print(f"total loss on the epoch : {str(sum(total_losses))}")
    FINE_TUNED_MODEL = output_dir + "/Swin_tiny_Object_Detection_Pascal_VOC_epoch_"+ str(i)+".pt"
    torch.save(model.state_dict(), FINE_TUNED_MODEL)

    return model, optimizer, scheduler

In [8]:
def compute_recall(ious, iou_threshold):
    """
    Compute recall for a batch of predicted boxes and GT boxes
    We assume first dimension corresponds to GT boxes and second dimension to predicted boxes.
    """

    ious = (ious > iou_threshold).int()

    # For each GT box, check if there is at least one predicted box with IoU above the threshold
    nb_of_gtbb_found = (ious.sum(dim=1) > 0).sum().item()
    nb_of_gt_boxes = ious.shape[0]
    return nb_of_gtbb_found, nb_of_gt_boxes


def compute_precision(ious, iou_threshold):
    """
    Compute precision for a batch of predicted boxes and GT boxes
    We assume first dimension corresponds to GT boxes and second dimension to predicted boxes.
    """

    ious = (ious > iou_threshold).int()

    # For each predicted box, check if there is at least one GT box with IoU above the threshold
    nb_of_pred_usefull = (ious.sum(dim=0) > 0).sum().item()
    nb_of_pred_boxes = ious.shape[1]
    return nb_of_pred_usefull, nb_of_pred_boxes


def compute_statistics_batch(confidence_score, ROIs, final_boxes, indice_class_selected, confidence_score_threshold, GT_bounding_boxes, GT_labels, list_labels, iou_threshold):
    from Swin_OD import SwinTransformerDetection
    from utils import IoU

    # remove boxes with confidence score below the threshold
    all_labels_confidence_score, all_labels_ROIs, all_labels_final_boxes, all_labels_indice_class_selected = SwinTransformerDetection.postprocess_remove_low_confidence_boxes(confidence_score_threshold, confidence_score, ROIs, final_boxes, indice_class_selected)

    batch_results_per_label = {}
    for label in list_labels:

        # Postprocess the predictions to gather by image and on the desired label
        confidence_score, ROIs, final_boxes, indice_class_selected = SwinTransformerDetection.postprocess_match_label(label, all_labels_confidence_score, all_labels_ROIs, all_labels_final_boxes, all_labels_indice_class_selected)
        _, pred_boxes, _ = SwinTransformerDetection.postprocess_gather_by_image(confidence_score, ROIs, final_boxes, indice_class_selected)

        batch_results_per_label[label] = {}
        batch_nb_pred_usefull = 0
        batch_nb_pred_boxes = 0
        batch_nb_gtbb_found = 0
        batch_nb_gt_boxes = 0
        for pred, gt_bb, gt_label in zip(pred_boxes, GT_bounding_boxes, GT_labels):

            # Remove the GT bb that do not correspond to the current label
            mask_gt = gt_label == label
            gt_bb = gt_bb[mask_gt]

            # Compute IoU between predicted boxes and GT boxes
            aligned_gt_bb = gt_bb.unsqueeze(1).expand(gt_bb.shape[0], pred.shape[0], 4)
            aligned_pred = pred.unsqueeze(0).expand(gt_bb.shape[0], pred.shape[0], 4)
            ious = IoU(aligned_gt_bb, aligned_pred)

            nb_of_pred_usefull, nb_of_pred_boxes = compute_precision(ious, iou_threshold)
            nb_of_gtbb_found, nb_of_gt_boxes = compute_recall(ious, iou_threshold)

            batch_nb_pred_usefull += nb_of_pred_usefull
            batch_nb_pred_boxes += nb_of_pred_boxes
            batch_nb_gtbb_found += nb_of_gtbb_found
            batch_nb_gt_boxes += nb_of_gt_boxes

        batch_results_per_label[label]['batch_nb_pred_usefull'] = batch_nb_pred_usefull
        batch_results_per_label[label]['batch_nb_pred_boxes'] = batch_nb_pred_boxes
        batch_results_per_label[label]['batch_nb_gtbb_found'] = batch_nb_gtbb_found
        batch_results_per_label[label]['batch_nb_gt_boxes'] = batch_nb_gt_boxes
    return batch_results_per_label


def display_metrics(global_results_per_label, mapping_id_class=None):
    global_nb_pred_usefull = 0
    global_nb_pred_boxes = 0
    global_nb_gtbb_found = 0
    global_nb_gt_boxes = 0

    for label, metrics in global_results_per_label.items():
        global_nb_pred_usefull += metrics['batch_nb_pred_usefull']
        global_nb_pred_boxes += metrics['batch_nb_pred_boxes']
        global_nb_gtbb_found += metrics['batch_nb_gtbb_found']
        global_nb_gt_boxes += metrics['batch_nb_gt_boxes']

    for label, metrics in global_results_per_label.items():
        if mapping_id_class is not None:
            classname = mapping_id_class[label]
        else:
            classname = ""

        precision = metrics['batch_nb_pred_usefull'] / metrics['batch_nb_pred_boxes'] if metrics['batch_nb_pred_boxes'] > 0 else 0
        recall = metrics['batch_nb_gtbb_found'] / metrics['batch_nb_gt_boxes'] if metrics['batch_nb_gt_boxes'] > 0 else 0
        print(f"Label {label}: {classname} - Precision: {precision:.4f}, Recall: {recall:.4f}")

    # Print global metrics
    global_precision = global_nb_pred_usefull / global_nb_pred_boxes if global_nb_pred_boxes > 0 else 0
    global_recall = global_nb_gtbb_found / global_nb_gt_boxes if global_nb_gt_boxes > 0 else 0
    print(f"Global - Precision: {global_precision:.4f}, Recall: {global_recall:.4f}")

In [9]:
def evaluate(i, model, device, validation_loader, confidence_score_threshold, list_labels, iou_threshold, mapping_id_class):
    model.eval()
    global_results_per_label = {}
    global_loss = []

    with torch.no_grad():
        for _, image_src, GT_bounding_boxes, GT_labels in validation_loader:
            image_src = image_src.to(device)
            GT_bounding_boxes = GT_bounding_boxes.to(device)
            GT_labels = GT_labels.to(device)

            with autocast(device_type=device, dtype=torch.bfloat16):
                loss, confidence_score, ROIs, final_boxes, indice_class_selected = model.forward_train(image_src, GT_bounding_boxes, GT_labels)

            global_loss.append(loss.item())
            if i % 5 == 0:
              batch_results_per_label = compute_statistics_batch(confidence_score, ROIs, final_boxes, indice_class_selected, confidence_score_threshold, GT_bounding_boxes, GT_labels, list_labels, iou_threshold)
              # Aggregate results globally
              for label in list_labels[1:]:
                  if label not in global_results_per_label:
                      global_results_per_label[label] = {'batch_nb_pred_usefull': 0, 'batch_nb_pred_boxes': 0, 'batch_nb_gtbb_found': 0, 'batch_nb_gt_boxes': 0}
                  for metric in global_results_per_label[label]:
                      global_results_per_label[label][metric] += batch_results_per_label[label][metric]

    print(f"Evaluation complet")
    if i % 10 == 0:
      display_metrics(global_results_per_label, mapping_id_class=mapping_id_class)
    print(f"Global loss : {str(sum(global_loss)/len(global_loss))}")
    return global_results_per_label, sum(global_loss)/len(global_loss)


## Warmup

In [12]:
n_epochs = 5

for p in model.backbone.parameters():
  p.requires_grad = False

head_params =[p for n, p in model.OD_head.named_parameters()]

optimizer = AdamW(model.OD_head.parameters(), lr=1e-5, weight_decay=0.05)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-4,
    steps_per_epoch=len(training_dataloader),
    epochs=n_epochs,
    pct_start=0.2,
    anneal_strategy='cos'
)

output_dir = "/content/drive/MyDrive/output_model"
os.makedirs(output_dir, exist_ok=True)

In [13]:
#track metrics
losses_val = []
training_stats = []
confidence_score_threshold = 0.5
list_labels = [i for i in range(20)]
iou_threshold = 0.75

model.train()
for i in range(n_epochs):
    print(f"Epoch {i+1}")
    model, optimizer, scheduler = train_one_epoch(i, model, str(device), training_dataloader, optimizer, scheduler, output_dir, accumulation_step=1)
    print(f"Evaluation ")
    training_stat, loss = evaluate(i, model, str(device), validating_dataloader, confidence_score_threshold, list_labels, iou_threshold, mapping_id_class)
    training_stats.append(training_stat)
    losses_val.append(loss)


Epoch 1
training loss mean on last 20 steps : 7.723931002616882
training loss mean on last 20 steps : 6.49890501499176
training loss mean on last 20 steps : 5.670726680755616
training loss mean on last 20 steps : 5.389799475669861
training loss mean on last 20 steps : 5.279393911361694
training loss mean on last 20 steps : 5.0816216468811035
training loss mean on last 20 steps : 4.996305632591247
training loss mean on last 20 steps : 4.937502861022949
training loss mean on last 20 steps : 4.946558284759521
training loss mean on last 20 steps : 4.947532606124878
total loss on the epoch : 1138.1985669136047
Evaluation 
Evaluation complet
Label 1: person - Precision: 0.0000, Recall: 0.0000
Label 2: bird - Precision: 0.0000, Recall: 0.0000
Label 3: cat - Precision: 0.0000, Recall: 0.0000
Label 4: cow - Precision: 0.0000, Recall: 0.0000
Label 5: dog - Precision: 0.0000, Recall: 0.0000
Label 6: horse - Precision: 0.0000, Recall: 0.0000
Label 7: sheep - Precision: 0.0000, Recall: 0.0000
Label

In [14]:
n_epochs = 100

for p in model.backbone.parameters():
  p.requires_grad = True

optimizer = AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-5},
    {'params': model.OD_head.parameters(), 'lr': 1e-4}
], weight_decay=0.1)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[1e-5, 1e-4],
    steps_per_epoch=len(training_dataloader),
    epochs=n_epochs,
    pct_start=0.2,
    anneal_strategy='cos'
)

output_dir = "/content/drive/MyDrive/output_model"
os.makedirs(output_dir, exist_ok=True)

In [15]:
#track metrics
losses_val = []
training_stats = []
confidence_score_threshold = 0.5
list_labels = [i for i in range(20)]
iou_threshold = 0.75

model.train()
for i in range(n_epochs):
    print(f"Epoch {i+1}")
    model, optimizer, scheduler = train_one_epoch(i, model, str(device), training_dataloader, optimizer, scheduler, output_dir, accumulation_step=1)
    print(f"Evaluation ")
    training_stat, loss = evaluate(i, model, str(device), validating_dataloader, confidence_score_threshold, list_labels, iou_threshold, mapping_id_class)
    training_stats.append(training_stat)
    losses_val.append(loss)


Epoch 1
training loss mean on last 20 steps : 4.639123439788818
training loss mean on last 20 steps : 4.526006698608398
training loss mean on last 20 steps : 4.674517583847046
training loss mean on last 20 steps : 4.5892175197601315
training loss mean on last 20 steps : 4.547904419898987
training loss mean on last 20 steps : 4.58558566570282
training loss mean on last 20 steps : 4.647281360626221
training loss mean on last 20 steps : 4.622970485687256
training loss mean on last 20 steps : 4.655994701385498
training loss mean on last 20 steps : 4.554394197463989
total loss on the epoch : 948.0151333808899
Evaluation 
Evaluation complet
Label 1: person - Precision: 0.0000, Recall: 0.0000
Label 2: bird - Precision: 0.0000, Recall: 0.0000
Label 3: cat - Precision: 0.0000, Recall: 0.0000
Label 4: cow - Precision: 0.0000, Recall: 0.0000
Label 5: dog - Precision: 0.0000, Recall: 0.0000
Label 6: horse - Precision: 0.0000, Recall: 0.0000
Label 7: sheep - Precision: 0.0000, Recall: 0.0000
Label 

KeyboardInterrupt: 

In [ ]:
model.load_state_dict(torch.load("/content/drive/MyDrive/output_model/Swin_tiny_Object_Detection_Pascal_VOC_epoch_85.pt"))

In [ ]:
n_epochs = 100

for p in model.backbone.parameters():
  p.requires_grad = True

optimizer = AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-5},
    {'params': model.OD_head.parameters(), 'lr': 1e-4}
], weight_decay=0.1)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[1e-5, 1e-4],
    steps_per_epoch=len(training_dataloader),
    epochs=n_epochs,
    pct_start=0.0,
    anneal_strategy='cos'
)

output_dir = "/content/drive/MyDrive/output_model"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
#track metrics
losses_val = []
training_stats = []
confidence_score_threshold = 0.5
list_labels = [i for i in range(20)]
iou_threshold = 0.75

model.train()
for i in range(100, 100 + n_epochs):
    print(f"Epoch {i+1}")
    model, optimizer, scheduler = train_one_epoch(i, model, str(device), training_dataloader, optimizer, scheduler, output_dir, accumulation_step=1)
    print(f"Evaluation ")
    training_stat, loss = evaluate(i, model, str(device), validating_dataloader, confidence_score_threshold, list_labels, iou_threshold, mapping_id_class)
    training_stats.append(training_stat)
    losses_val.append(loss)


In [10]:
model.load_state_dict(torch.load("/content/drive/MyDrive/output_model/Swin_tiny_Object_Detection_Pascal_VOC_epoch_13.pt"))

<All keys matched successfully>

In [13]:
confidence_score_threshold = 0.5
list_labels = [i for i in range(20)]
evaluate(10, model, str(device), validating_dataloader, 0.5, list_labels, 0.75, mapping_id_class)

Evaluation complet
Label 1: person - Precision: 0.0000, Recall: 0.0000
Label 2: bird - Precision: 0.0000, Recall: 0.0000
Label 3: cat - Precision: 0.0000, Recall: 0.0000
Label 4: cow - Precision: 0.0000, Recall: 0.0000
Label 5: dog - Precision: 0.0000, Recall: 0.0000
Label 6: horse - Precision: 0.0000, Recall: 0.0000
Label 7: sheep - Precision: 0.0000, Recall: 0.0000
Label 8: aeroplane - Precision: 0.0000, Recall: 0.0000
Label 9: bicycle - Precision: 0.0000, Recall: 0.0000
Label 10: boat - Precision: 0.0000, Recall: 0.0000
Label 11: bus - Precision: 0.0000, Recall: 0.0000
Label 12: car - Precision: 0.0000, Recall: 0.0000
Label 13: motorbike - Precision: 0.0000, Recall: 0.0000
Label 14: train - Precision: 0.0000, Recall: 0.0000
Label 15: bottle - Precision: 0.0000, Recall: 0.0000
Label 16: chair - Precision: 0.0000, Recall: 0.0000
Label 17: diningtable - Precision: 0.0000, Recall: 0.0000
Label 18: pottedplant - Precision: 0.0000, Recall: 0.0000
Label 19: sofa - Precision: 0.0000, Recall

({1: {'batch_nb_pred_usefull': 0,
   'batch_nb_pred_boxes': 36,
   'batch_nb_gtbb_found': 0,
   'batch_nb_gt_boxes': 2893},
  2: {'batch_nb_pred_usefull': 0,
   'batch_nb_pred_boxes': 0,
   'batch_nb_gtbb_found': 0,
   'batch_nb_gt_boxes': 338},
  3: {'batch_nb_pred_usefull': 0,
   'batch_nb_pred_boxes': 0,
   'batch_nb_gtbb_found': 0,
   'batch_nb_gt_boxes': 335},
  4: {'batch_nb_pred_usefull': 0,
   'batch_nb_pred_boxes': 0,
   'batch_nb_gtbb_found': 0,
   'batch_nb_gt_boxes': 184},
  5: {'batch_nb_pred_usefull': 0,
   'batch_nb_pred_boxes': 0,
   'batch_nb_gtbb_found': 0,
   'batch_nb_gt_boxes': 436},
  6: {'batch_nb_pred_usefull': 0,
   'batch_nb_pred_boxes': 0,
   'batch_nb_gtbb_found': 0,
   'batch_nb_gt_boxes': 230},
  7: {'batch_nb_pred_usefull': 0,
   'batch_nb_pred_boxes': 0,
   'batch_nb_gtbb_found': 0,
   'batch_nb_gt_boxes': 238},
  8: {'batch_nb_pred_usefull': 0,
   'batch_nb_pred_boxes': 12,
   'batch_nb_gtbb_found': 0,
   'batch_nb_gt_boxes': 251},
  9: {'batch_nb_pred_

In [14]:
confidence_score_threshold = 0.5
list_labels = [i for i in range(20)]
evaluate(10, model, str(device), validating_dataloader, confidence_score_threshold, list_labels, 0.5, mapping_id_class)

Evaluation complet
Label 1: person - Precision: 0.0000, Recall: 0.0000
Label 2: bird - Precision: 0.0000, Recall: 0.0000
Label 3: cat - Precision: 0.0000, Recall: 0.0000
Label 4: cow - Precision: 0.0000, Recall: 0.0000
Label 5: dog - Precision: 0.0000, Recall: 0.0000
Label 6: horse - Precision: 0.0000, Recall: 0.0000
Label 7: sheep - Precision: 0.0000, Recall: 0.0000
Label 8: aeroplane - Precision: 0.0000, Recall: 0.0000
Label 9: bicycle - Precision: 0.0000, Recall: 0.0000
Label 10: boat - Precision: 0.0000, Recall: 0.0000
Label 11: bus - Precision: 0.0000, Recall: 0.0000
Label 12: car - Precision: 0.0000, Recall: 0.0000
Label 13: motorbike - Precision: 0.0000, Recall: 0.0000
Label 14: train - Precision: 0.0000, Recall: 0.0000
Label 15: bottle - Precision: 0.0000, Recall: 0.0000
Label 16: chair - Precision: 0.0000, Recall: 0.0000
Label 17: diningtable - Precision: 0.0000, Recall: 0.0000
Label 18: pottedplant - Precision: 0.0000, Recall: 0.0000
Label 19: sofa - Precision: 0.0000, Recall

({1: {'batch_nb_pred_usefull': 0,
   'batch_nb_pred_boxes': 41,
   'batch_nb_gtbb_found': 0,
   'batch_nb_gt_boxes': 2893},
  2: {'batch_nb_pred_usefull': 0,
   'batch_nb_pred_boxes': 0,
   'batch_nb_gtbb_found': 0,
   'batch_nb_gt_boxes': 338},
  3: {'batch_nb_pred_usefull': 0,
   'batch_nb_pred_boxes': 0,
   'batch_nb_gtbb_found': 0,
   'batch_nb_gt_boxes': 335},
  4: {'batch_nb_pred_usefull': 0,
   'batch_nb_pred_boxes': 0,
   'batch_nb_gtbb_found': 0,
   'batch_nb_gt_boxes': 184},
  5: {'batch_nb_pred_usefull': 0,
   'batch_nb_pred_boxes': 0,
   'batch_nb_gtbb_found': 0,
   'batch_nb_gt_boxes': 436},
  6: {'batch_nb_pred_usefull': 0,
   'batch_nb_pred_boxes': 0,
   'batch_nb_gtbb_found': 0,
   'batch_nb_gt_boxes': 230},
  7: {'batch_nb_pred_usefull': 0,
   'batch_nb_pred_boxes': 0,
   'batch_nb_gtbb_found': 0,
   'batch_nb_gt_boxes': 238},
  8: {'batch_nb_pred_usefull': 0,
   'batch_nb_pred_boxes': 5,
   'batch_nb_gtbb_found': 0,
   'batch_nb_gt_boxes': 251},
  9: {'batch_nb_pred_u

# Debug

In [ ]:
model.load_state_dict(torch.load("/content/drive/MyDrive/output_model/Swin_tiny_Object_Detection_Pascal_VOC_epoch_85.pt"))

In [ ]:
model.train()
iter_ = iter(testing_dataloader)
next(iter_)
with torch.no_grad():
  _, image_src, GT_bounding_boxes, GT_labels = next(iter_)
  image_src = image_src.to(device)
  GT_bounding_boxes = GT_bounding_boxes.to(device)
  GT_labels = GT_labels.to(device)
  loss, confidence_score, ROIs, final_boxes, indice_class_selected = model.forward_train(image_src, GT_bounding_boxes, GT_labels)

In [ ]:
model.eval()
iter_ = iter(testing_dataloader)
next(iter_)
with torch.no_grad():
  _, image_src, GT_bounding_boxes, GT_labels = next(iter_)
  image_src = image_src.to(device)
  GT_bounding_boxes = GT_bounding_boxes.to(device)
  GT_labels = GT_labels.to(device)
  loss, confidence_score, ROIs, final_boxes, indice_class_selected = model.forward_train(image_src, GT_bounding_boxes, GT_labels)
  pred_confidence_score, pred_ROIs, pred_final_boxes, pred_indice_class_selected = model.forward(image_src)

print(GT_bounding_boxes)
print(confidence_score.shape)
print([ROI.shape for ROI in ROIs])
print(final_boxes.shape)
print(indice_class_selected.shape)
print("###############")
all_labels_confidence_score, all_labels_ROIs, all_labels_final_boxes, all_labels_indice_class_selected = SwinTransformerDetection.postprocess_remove_low_confidence_boxes(0.5, confidence_score, ROIs, final_boxes, indice_class_selected)
print(all_labels_confidence_score.shape)
print([ROI.shape for ROI in all_labels_ROIs])
print(all_labels_final_boxes.shape)
print(all_labels_indice_class_selected.shape)
print("###############")
nob_all_labels_confidence_score, nob_all_labels_ROIs, nob_all_labels_final_boxes, nob_all_labels_indice_class_selected = SwinTransformerDetection.postprocess_exclude_label(0, all_labels_confidence_score, all_labels_ROIs, all_labels_final_boxes, all_labels_indice_class_selected)
print(nob_all_labels_confidence_score.shape)
print([ROI.shape for ROI in all_labels_ROIs])
print(nob_all_labels_final_boxes.shape)
print(nob_all_labels_indice_class_selected.shape)
print("###############")
print(pred_confidence_score.shape)
print([ROI.shape for ROI in pred_ROIs])
print(pred_final_boxes.shape)
print(pred_indice_class_selected.shape)
print("###############")
pred_all_labels_confidence_score, pred_all_labels_ROIs, pred_all_labels_final_boxes, pred_all_labels_indice_class_selected = SwinTransformerDetection.postprocess_remove_low_confidence_boxes(0.5, pred_confidence_score, pred_ROIs, pred_final_boxes, pred_indice_class_selected)
print(pred_all_labels_confidence_score.shape)
print([ROI.shape for ROI in pred_all_labels_ROIs])
print(pred_all_labels_final_boxes.shape)
print(pred_all_labels_indice_class_selected.shape)
print("###############")
pred_nob_all_labels_confidence_score, pred_nob_all_labels_ROIs, pred_nob_all_labels_final_boxes, pred_nob_all_labels_indice_class_selected = SwinTransformerDetection.postprocess_exclude_label(0, pred_all_labels_confidence_score, pred_all_labels_ROIs, pred_all_labels_final_boxes, pred_all_labels_indice_class_selected)
print(pred_nob_all_labels_confidence_score.shape)
print([ROI.shape for ROI in pred_nob_all_labels_ROIs])
print(pred_nob_all_labels_final_boxes.shape)
print(pred_nob_all_labels_indice_class_selected.shape)


In [ ]:
def show_image(image, bounding_boxes, labels, mapping_id_class):
    # 1. Handle PyTorch Image Tensor format (C, H, W) -> (H, W, C)
    if isinstance(image, torch.Tensor):
        # Move to CPU, detach from graph, and reorder dimensions for Matplotlib
        image = image.detach().cpu().permute(1, 2, 0).numpy()

        # If your image pixels are in range [0, 1], Matplotlib handles it fine.
        # If they are normalized with ImageNet mean/std, they might look washed out,
        # but the boxes will still align perfectly!

    # Convert boxes and labels to CPU numpy arrays if they are tensors
    if isinstance(bounding_boxes, torch.Tensor):
        bounding_boxes = bounding_boxes.detach().cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.detach().cpu().numpy()

    print(f"Image shape: {image.shape}, Boxes shape: {bounding_boxes.shape}, Labels shape: {labels.shape}")

    # Create a figure and axis
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image)

    # Draw bounding boxes
    for box, label in zip(bounding_boxes, labels):
        # 2. Pascal VOC standard format is [xmin, ymin, xmax, ymax]
        ymin, xmin, height, width = box

        # Get the string class name
        class_name = mapping_id_class.get(int(label), f"Class {int(label)}")

        # Create a rectangle patch
        rect = patches.Rectangle((xmin, ymin), width, height,
                                 linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)

        # 3. Add text label slightly above the bounding box
        ax.text(xmin, ymin - 5, class_name, color='white', fontsize=12, fontweight='bold',
                bbox=dict(facecolor='red', alpha=0.5, edgecolor='none', pad=2))

    # Hide axis and show image
    ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
def denormalize_and_convert_to_pil(tensor):

    img = tensor.clone()
    # img = img * std + mean  # Undo normalization -> back to [0,1]
    img = img.clamp(0, 1)  # Ensure range is valid
    img = img.mul(255).byte()  # Scale to [0,255] and convert to uint8
    img = img.permute(1, 2, 0)  # Convert from CxHxW to HxWxC for PIL
    return img.cpu().numpy()

In [ ]:
indice = 0
# Simulate data from the dataloader
filename, image, bounding_boxes, labels = next(iter(testing_dataloader))


# Convert image tensor to numpy and scale to [0, 255]
image = (image[indice].permute(1, 2, 0).numpy() * 255).astype(np.uint8)

# Apply mask to remove invalid bounding boxes
mask_padding = labels[indice] != -1
bounding_boxes = bounding_boxes[indice][mask_padding]
labels = labels[indice][mask_padding]
show_image(image, bounding_boxes, labels, mapping_id_class)


In [ ]:
indice = 1
model.eval()
# Simulate data from the dataloader
image, bounding_boxes, labels = next(iter(training_dataloader))

confidence_score, ROIs, pred_boxes, labels = model.forward(image)

print(confidence_score.shape)
print([ROI.shape for ROI in ROIs])
print([pred_boxe.shape for pred_boxe in pred_boxes])
print([label.shape for label in labels])

In [ ]:
indice = 5
model.eval()
# Simulate data from the dataloader
filename, image, bounding_boxes, labels = next(iter(training_dataloader))
image = image.to(device)
bounding_boxes = bounding_boxes.to(device)
labels = labels.to(device)
with torch.no_grad():
  confidence_score, ROIs, pred_boxes, labels = model.forward(image)

# Convert image tensor to numpy and scale to [0, 255]
image = (image[indice].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)

# Apply mask to remove invalid bounding boxes
mask_padding = labels[indice] != -1
pred_box = pred_boxes[indice][mask_padding]
print(pred_box.shape)
label = labels[indice][mask_padding]
show_image(image, pred_box, label, mapping_id_class)


In [ ]:
for name, param in model.backbone.named_parameters():
    param.requires_grad = False

In [ ]:
for name, param in model.OD_head.detector.named_parameters():
    param.requires_grad = False

In [ ]:
for name, param in model.named_parameters():
    print(name, param.requires_grad)